In [ ]:
# Carga inicial del dataset

import pandas as pd

# Carga del dataset
df = pd.read_csv(r'C:\Users\gross\Documents\Proyectos\customer-churn-ml\data\raw\customer_churn_historical.csv')

print(df) , df.shape

     customerID  gender  SeniorCitizen Partner Dependents  tenure  \
0     6684-LWVH    Male              0      No         No       4   
1     7027-JIFO    Male              0      No         No      35   
2     5981-VOQO  Female              0      No         No      45   
3     7266-HYDM    Male              0     Yes         No      31   
4     2821-JDLS  Female              0     Yes         No       5   
...         ...     ...            ...     ...        ...     ...   
7038  5878-LZXJ  Female              1     Yes        Yes      41   
7039  8490-ENJV  Female              0     Yes         No       2   
7040  7152-RWIV  Female              0      No         No       5   
7041  5768-FYEE    Male              1      No         No      25   
7042  1398-XMGM    Male              0     Yes        Yes      25   

     PhoneService MultipleLines InternetService       OnlineSecurity  ...  \
0             Yes           Yes              No  No internet service  ...   
1             Yes

(None, (7043, 21))

In [ ]:
# Tipos de datos

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [ ]:
# valores nulos por columna

df.isnull().sum()


customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        26
Churn                0
dtype: int64

In [4]:
# Distribución del target

df['Churn'].value_counts()

Churn
No     5186
Yes    1857
Name: count, dtype: int64

In [5]:
# Distribución del target en porcentaje

df['Churn'].value_counts(normalize=True) * 100

Churn
No     73.633395
Yes    26.366605
Name: proportion, dtype: float64

In [6]:
# Cardinalidad de las columnas categóricas (para detectar anomalías)

categorical_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} valores únicos -> {df[col].unique()}")

gender: 2 valores únicos -> <StringArray>
['Male', 'Female']
Length: 2, dtype: str
Partner: 2 valores únicos -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Dependents: 2 valores únicos -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
PhoneService: 2 valores únicos -> <StringArray>
['Yes', 'No']
Length: 2, dtype: str
MultipleLines: 3 valores únicos -> <StringArray>
['Yes', 'No', 'No phone service']
Length: 3, dtype: str
InternetService: 3 valores únicos -> <StringArray>
['No', 'DSL', 'Fiber optic']
Length: 3, dtype: str
OnlineSecurity: 3 valores únicos -> <StringArray>
['No internet service', 'No', 'Yes']
Length: 3, dtype: str
OnlineBackup: 3 valores únicos -> <StringArray>
['No internet service', 'No', 'Yes']
Length: 3, dtype: str
DeviceProtection: 3 valores únicos -> <StringArray>
['No internet service', 'No', 'Yes']
Length: 3, dtype: str
TechSupport: 3 valores únicos -> <StringArray>
['No internet service', 'No', 'Yes']
Length: 3, dtype: str
StreamingTV: 3 valores únicos ->

C:\Users\gross\AppData\Local\Temp\ipykernel_7664\3670242725.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])


In [ ]:
# Prueba del pipeline de preprocessing

import sys
sys.path.append('..')  # para que encuentre el módulo src desde notebooks/

from src.features.build_pipeline import build_preprocessor

prep = build_preprocessor()
print(prep)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
            

In [ ]:
# Carga de datos vía función

from src.data.load_data import load_data, split_data

df = load_data('../data/raw/customer_churn_historical.csv')
X_train, X_test, y_train, y_test = split_data(df)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (5634, 19) (5634,)
Test: (1409, 19) (1409,)


In [ ]:
# Proporciones train/test

print("Proporción Churn en train:")
print(y_train.value_counts(normalize=True))
print("\nProporción Churn en test:")
print(y_test.value_counts(normalize=True))

Proporción Churn en train:
Churn
No     0.736422
Yes    0.263578
Name: proportion, dtype: float64

Proporción Churn en test:
Churn
No     0.735983
Yes    0.264017
Name: proportion, dtype: float64


In [ ]:
# Entrenar y evaluar el baseline

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

from src.features.build_pipeline import build_preprocessor

# Baseline: modelo simple que predice siempre la clase mas comun
pipeline_baseline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', DummyClassifier(strategy='most_frequent'))
])

pipeline_baseline.fit(X_train, y_train)
pred_baseline = pipeline_baseline.predict(X_test)
proba_baseline = pipeline_baseline.predict_proba(X_test)[:, 1]

print("Precision:", round(precision_score(y_test, pred_baseline, pos_label='Yes'), 3))
print("Recall:", round(recall_score(y_test, pred_baseline, pos_label='Yes'), 3))
print("F1:", round(f1_score(y_test, pred_baseline, pos_label='Yes'), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba_baseline), 3))
print(confusion_matrix(y_test, pred_baseline, labels=['No', 'Yes']))

Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.5
[[1037    0]
 [ 372    0]]


c:\Users\gross\Documents\Proyectos\customer-churn-ml\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [15]:
# Regresion Logística

from sklearn.linear_model import LogisticRegression

pipeline_logistic = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline_logistic.fit(X_train, y_train)
pred_logistic = pipeline_logistic.predict(X_test)
proba_logistic = pipeline_logistic.predict_proba(X_test)[:, 1]

print("Precision:", round(precision_score(y_test, pred_logistic, pos_label='Yes'), 3))
print("Recall:", round(recall_score(y_test, pred_logistic, pos_label='Yes'), 3))
print("F1:", round(f1_score(y_test, pred_logistic, pos_label='Yes'), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba_logistic), 3))
print(confusion_matrix(y_test, pred_logistic, labels=['No', 'Yes']))

Precision: 0.664
Recall: 0.452
F1: 0.538
ROC-AUC: 0.812
[[952  85]
 [204 168]]


In [16]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

pipeline_forest = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', RandomForestClassifier(random_state=42))
])

pipeline_forest.fit(X_train, y_train)
pred_forest = pipeline_forest.predict(X_test)
proba_forest = pipeline_forest.predict_proba(X_test)[:, 1]

print("Precision:", round(precision_score(y_test, pred_forest, pos_label='Yes'), 3))
print("Recall:", round(recall_score(y_test, pred_forest, pos_label='Yes'), 3))
print("F1:", round(f1_score(y_test, pred_forest, pos_label='Yes'), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba_forest), 3))
print(confusion_matrix(y_test, pred_forest, labels=['No', 'Yes']))

Precision: 0.631
Recall: 0.414
F1: 0.5
ROC-AUC: 0.794
[[947  90]
 [218 154]]
